# GPPO Colab 一键机制验证

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Battleplus/GPPO/blob/8.8-GPPO%E6%97%A0%E5%81%8F%E5%A5%BD/colab/GPPO_Colab_One_Click.ipynb)

选择 **运行时 → 全部运行** 即可。Notebook 会自动完成环境检查、CPU/GPU 基准、六模型训练、固定 test100、Random/Greedy、Event/Full 重放、中文报告和 ZIP 打包。

> 推荐运行时：Python 3 + L4 GPU。结果默认保存到 Google Drive，Colab 断线后重新全部运行即可续跑。

## 1. 参数

通常无需修改。`CONCURRENT_JOBS=2` 表示最多同时训练两个模型；如果 Colab 只分配了一个 CPU 核，可改为 `1`。

In [ ]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/Battleplus/GPPO.git"
BRANCH = "8.8-GPPO无偏好"
REPOSITORY_DIR = Path("/content/GPPO")
OUTPUT_ROOT = Path("/content/drive/MyDrive/GPPO_one_click")
CONCURRENT_JOBS = 2
AUTO_DOWNLOAD_ZIP = True

print({
    "branch": BRANCH,
    "repository": str(REPOSITORY_DIR),
    "output": str(OUTPUT_ROOT),
    "concurrent_jobs": CONCURRENT_JOBS,
})

## 2. 挂载 Google Drive

首次运行时按提示授权。训练 checkpoint、日志和最终报告都会持久化到 Drive。

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("该 Notebook 应在 Google Colab 中运行。") from exc

drive.mount("/content/drive")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive 已挂载：", OUTPUT_ROOT)

## 3. 克隆或更新代码

如果代码目录已经存在，则使用 fast-forward 更新，不删除已有本地结果。

In [ ]:
import subprocess

def run_command(command, cwd=None, env=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

if (REPOSITORY_DIR / ".git").exists():
    run_command(["git", "fetch", "origin", BRANCH], cwd=REPOSITORY_DIR)
    run_command(["git", "checkout", BRANCH], cwd=REPOSITORY_DIR)
    run_command(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPOSITORY_DIR)
else:
    run_command([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPOSITORY_URL, str(REPOSITORY_DIR),
    ])

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR, text=True
).strip()
print("当前 commit：", commit)

## 4. 硬件检查

这里只记录 Colab 实际分配的硬件。后续脚本会用端到端训练时间自动决定 CPU 或 CUDA。

In [ ]:
import json
import os
import platform

try:
    gpu_description = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
    ).strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu_description = None

hardware = {
    "python": platform.python_version(),
    "cpu_count": os.cpu_count(),
    "gpu": gpu_description,
}
print(json.dumps(hardware, indent=2, ensure_ascii=False))

## 5. 一键运行全部机制验证任务

本单元格会持续运行较长时间。包含六个可学习模型，并自动完成评估和报告。若运行时中断，重新选择 **全部运行** 即可从 Drive checkpoint 接续。

In [ ]:
import os

training_environment = os.environ.copy()
training_environment.update({
    "OUTPUT_ROOT": str(OUTPUT_ROOT),
    "ONE_CLICK_JOBS": str(CONCURRENT_JOBS),
    "AUTO_DOWNLOAD": "0",
})

run_command(
    ["bash", "colab/run_everything_once.sh"],
    cwd=REPOSITORY_DIR,
    env=training_environment,
)
print("全部训练与评估任务已经完成。")

## 6. 显示结果

显示最终中文报告和机器可读结果。请重点检查 GPPO/PPO、Adaptive/NoGate/SingleHead 以及 Event/Full。

In [ ]:
from IPython.display import Markdown, display

report_path = OUTPUT_ROOT / "quick_seed1_100" / "QUICK_MECHANISM_REPORT_ZH.md"
result_path = OUTPUT_ROOT / "quick_seed1_100" / "quick_mechanism_result.json"
manifest_path = OUTPUT_ROOT / "ONE_CLICK_MANIFEST.json"
archive_path = Path(f"{OUTPUT_ROOT}.zip")

for required_path in (report_path, result_path, manifest_path, archive_path):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

display(Markdown(report_path.read_text(encoding="utf-8")))
result_payload = json.loads(result_path.read_text(encoding="utf-8"))
print(json.dumps({
    "decision": result_payload.get("decision"),
    "checks": result_payload.get("checks"),
    "archive": str(archive_path),
}, indent=2, ensure_ascii=False))

## 7. 下载 ZIP

如果 `AUTO_DOWNLOAD_ZIP=True`，运行到这里会自动开始下载。ZIP 同时保留在 Google Drive。

In [ ]:
if AUTO_DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(archive_path))
else:
    print("ZIP 已保存：", archive_path)

## 验收边界

本 Notebook 完成的是 `T5-10-48 / seed1 / 100 iterations / 六模型 / test100` 快速机制验证，用于决定是否继续正式五种子训练。它不能单独证明 GPPO 稳定优于 PPO，也不能替代四规模五种子的论文级统计。